### BayesianRidge回帰の簡単な例

scikit learnの
BayesianRidge回帰
を用います。



In [ ]:
from sklearn import linear_model
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
def get_data():
    """make small data.

    Returns:
        np.ndarray: X,
        np.ndarray: T,
        np.ndarray: Y0,
        np.ndarray: Yerr.
    """
    # Xの範囲
    Xrange = [-1, 1]
    # 表示するwの範囲
    drange = [[-0.5, -0.1], [-0.1, 0.3]]

    # 解の構成
    # w
    w = np.array([-0.3, 0.1])

    # number of all the points
    N = 30

    # noise ,  std dev of gaussian
    sigma = 0.05

    X1 = np.linspace(Xrange[0], Xrange[1], N)
    X2 = np.sin(10*X1)
    X = np.vstack([X1, X2]).T
    # print(X.shape)
    Y0 = w[0]*X[:, 0]+w[1]*X[:, 1]

    T = np.random.normal(Y0, scale=sigma, size=N)
    Yerr = sigma

    # 表示
    fig, ax = plt.subplots()
    ax.plot(X[:, 0], Y0, ".-", label="Y0")
    ax.errorbar(X[:, 0], T, Yerr, fmt=".-", label="T, bar=std dev.")
    ax.legend()

    return X, T, Y0, Yerr


g_X, g_T, g_Y0, g_Yerr = get_data()


randomに5点だけ取る。

In [ ]:
import numpy as np
idxall = list(range(g_X.shape[0]))
n_sample = 5
g_idx = np.random.permutation(idxall)[:n_sample]
g_idx


In [ ]:
g_reg = linear_model.BayesianRidge()
g_Xtrain = g_X[g_idx]
g_Ttrain = g_T[g_idx]
g_reg.fit(g_Xtrain, g_Ttrain)

g_Yp, g_std = g_reg.predict(g_X, return_std=True)
g_reg.coef_, g_reg.sigma_


上が係数のmeanとcovarenceを与えます。

ハイパーパラメタが下になります。

In [ ]:
g_reg.lambda_, g_reg.alpha_


Yp,stdが予測値の平均と標準偏差であるのでplotしてみます。

In [ ]:
import matplotlib.pyplot as plt


def show_Xyyerror(X, T, Y0, Yp, std, idx):
    """show error plot.

    Args:
        X (np.ndarray): X.
        T (np.ndarray): T.
        Y0 (np.ndarray): Y0.
        Yp (np.ndarray): predicted Y.
        std (np.ndarray): observation error of Y.
        idx ([int]): indeces of observations.
    """
    fig, ax = plt.subplots()
    ax.plot(X[idx, 0], T[idx], "o", label="training")
    ax.plot(X[:, 0], Y0, color="blue", label="Y0")

    ax.errorbar(X[:, 0], Yp, std, color="orange", label="predict(+-sigma)")
    ax.legend()


show_Xyyerror(g_X, g_T, g_Y0, g_Yp, g_std, g_idx)


stdは小さいのですが、Xによる定量的な違いがあります。

In [ ]:
plt.plot(g_X[:, 0], g_std, "o-")



reg.sigma_に共分散行列があるので
multivariate_normalで乱数から分布を可視化することが可能です。

In [ ]:
from numpy.random import multivariate_normal


def plot_Xy_rand(X, Y0, T, reg, idx):
    """plot Xy of models.

    Args:
        X (np.ndarray): X.
        Y0 (np.ndarray): Y0.
        T (np.ndarray): T.
        reg (BayesianRidge): regression model.
        idx ([int]): a list of training data.
    """
    fig, ax = plt.subplots()
    ax.plot(X[idx, 0], T[idx], "o", color="blue", label="selected")
    ax.plot(X[:, 0], Y0, label="Y0", linewidth=1, color="blue")

    w = reg.coef_
    Y_rand = w[0]*X[:, 0]+w[1]*X[:, 1]
    ax.plot(X[:, 0], Y_rand, "-", color="red", label="predict mean")

    n = 100
    w_rand = multivariate_normal(reg.coef_, reg.sigma_, size=n)
    for i, w_rand1 in enumerate(w_rand):
        # print("w",w_rand1)
        Y_rand = w_rand1[0]*X[:, 0]+w_rand1[1]*X[:, 1]
        ax.plot(X[:, 0], Y_rand, "-", linewidth=1, color="red", alpha=0.05)
    ax.legend()


plot_Xy_rand(g_X, g_Y0, g_T, g_reg, g_idx)


ではこの手法から各点の分布を求めてみます。

まず、分布データを貯める。

In [ ]:
def make_Y_distribution(X, reg):
    """make a set of Y.
    
    Args:
        X (np.ndarray): X.
        reg (BayesianRidge): model.

    Returns:
        np.ndarray: a set of Y.
    """
    Y_distribution = []
    n_ysample = 5000
    w_rand = multivariate_normal(reg.coef_, reg.sigma_, size=n_ysample)
    for i, w_rand1 in enumerate(w_rand):
        # print("w",w_rand1)
        Y_rand = w_rand1[0]*X[:, 0]+w_rand1[1]*X[:, 1]
        Y_distribution.append(Y_rand)
    # transpose
    Y_distribution = np.array(Y_distribution).T
    return Y_distribution


g_Y_distribution = make_Y_distribution(g_X, g_reg)


各点毎にn_ysample個の分布データがある。

gaussianをGMMでfitする。

In [ ]:
from sklearn.mixture import GaussianMixture


In [ ]:
def make_means_covar(Y_distribution):
    """make mean values and covaraiance matrix from Y_distribution.
    
    Args:
        Y_distribution (np.ndarray): a set of Y.
    """
    y_means = []
    y_covariances = []
    for y in Y_distribution:
        # plt.hist(y,bins=20)
        # plt.show()
        gmm = GaussianMixture(n_components=1)
        # gmmが二次元配列しか受け付けないので yを二次元配列(N,1)に直す。
        gmm.fit(y.reshape(-1, 1))
        # print(gmm.means_,gmm.covariances_)
        y_means.append(gmm.means_.ravel())
        y_covariances.append(gmm.covariances_.ravel())

    y_means = np.array(y_means)
    y_covariances = np.array(y_covariances)
    return y_means, y_covariances


g_ymeans, g_y_covariances = make_means_covar(g_Y_distribution)


n_sample点分のmeanとcovarianceが出たので再びplotする。

In [ ]:
from numpy.random import multivariate_normal


def plot_Xy_covar(X, Y0, T, y_means, y_covariances, idx):
    """plot X y and covariances.

    Args:
        X (np.ndarray): X.
        Y0 (np.ndarray): Y0.
        T (np.ndarray): T.
        y_means (np.ndarray): mean values of y.
        y_covariances (np.ndarray): covariances of y.
        idx ([int]): a list of training indeces.
    """
    fig, ax = plt.subplots()
    ax.plot(X[idx, 0], T[idx], "o", color="blue", label="selected")
    ax.plot(X[:, 0], Y0, label="Y0", linewidth=1, color="blue")

    y_std = np.sqrt(y_covariances)
    yplus = y_means + y_std
    yminus = y_means - y_std
    # yplus, yminusは二次元配列。
    # fill_betweenは一次元配列でないといけないので配列の次元を再び一次元に直す。
    ax.fill_between(X[:, 0], yminus.ravel(),
                    yplus.ravel(), color="red", alpha=0.3)
    ax.plot(X[:, 0], y_means.ravel(), color="red")
    ax.legend()
    fig.show()


plot_Xy_covar(g_X, g_Y0, g_T, g_ymeans, g_y_covariances, g_idx)
